In [ ]:
import os
import json
import time
import logging
import boto3
import faiss
import numpy as np
from typing import List

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# AWS Clients
s3_client = boto3.client("s3")
bedrock = boto3.client("bedrock-runtime")

# Configuration Dictionary
CONFIG = {
    "bucket_name": "sagemaker-us-east-1-188494237500",
    "json_folder": "dev/json",
    "index_folder": "dev/idx",
    "nlist": 512,
    "nprobe": 16,
    "num_training_samples": 1000,
    "embedding_models": {
        "amazon.titan-embed-text-v2:0": 1536,
        "anthropic.claude-3-sonnet-20240229-v1:0": 8192,
        "meta.llama3-70b-instruct-v1:0": 8192
    },
    "default_model": "amazon.titan-embed-text-v2:0"
}

query_cache = {}  # Embedding cache

# ✅ Read JSON file from S3 (Sequential)
def read_s3_json(bucket: str, file_key: str):
    try:
        response = s3_client.get_object(Bucket=bucket, Key=file_key)
        file_content = response["Body"].read().decode("utf-8").strip()
        return json.loads(file_content) if file_content else None
    except json.JSONDecodeError:
        logger.error(f"❌ Invalid JSON format in {file_key}.")
    except Exception as e:
        logger.error(f"❌ Failed to read {file_key}: {e}")
    return None

# ✅ Save JSON back to S3 (Sequential)
def save_s3_json(bucket: str, file_key: str, data):
    try:
        json_body = json.dumps(data, ensure_ascii=False).encode("utf-8")
        s3_client.put_object(Bucket=bucket, Key=file_key, Body=json_body)
        logger.info(f"✅ Successfully updated {file_key}")
    except Exception as e:
        logger.error(f"❌ Failed to save {file_key}: {e}")

# ✅ Clean text before embedding (Simple Sequential Processing)
def clean_text(text: str) -> str:
    """Cleans extracted text by removing unwanted characters and normalizing spaces."""
    return text.replace('\n', ' ').replace('\t', ' ').strip()

# ✅ Generate Embedding (Sequential - No Threads)
def generate_embedding(text, model_name, max_retries=1):
    """
    Generates an embedding using AWS Bedrock.
    - Checks the cache to avoid redundant API calls.
    - Handles missing or malformed responses properly.
    """
    cache_key = (text, model_name)
    
    if cache_key in query_cache:
        logger.info(f"⚡ Using cached embedding for '{text}' ({model_name})")
        return query_cache[cache_key]

    for attempt in range(max_retries + 1):
        try:
            logger.info(f"⚡ Requesting embedding from AWS Bedrock for {model_name} (Attempt {attempt + 1})...")
            start_time = time.time()

            # ✅ Make the request
            response = bedrock.invoke_model(
                modelId=model_name,
                contentType="application/json",
                accept="application/json",
                body=json.dumps({"inputText": text})
            )

            elapsed = time.time() - start_time
            logger.info(f"🕒 Bedrock embedding request took {elapsed:.2f} seconds.")

            # ✅ Validate response
            if "Body" not in response:
                logger.error(f"❌ AWS Bedrock response for {model_name} is missing 'Body'. Retrying...")
                continue

            # ✅ Read response
            response_body_bytes = response["Body"].read()
            if not response_body_bytes:
                logger.error(f"❌ AWS Bedrock response for {model_name} has an empty 'Body'. Retrying...")
                continue

            response_body = json.loads(response_body_bytes.decode("utf-8"))
            if "embedding" not in response_body:
                logger.error(f"❌ 'embedding' key missing in Bedrock response for {model_name}. Retrying...")
                continue

            embedding = response_body["embedding"]

            # ✅ Check embedding dimension
            expected_dim = CONFIG["embedding_models"].get(model_name, None)
            if expected_dim and len(embedding) != expected_dim:
                logger.warning(f"⚠️ Embedding dimension mismatch for {model_name}: Expected {expected_dim}, got {len(embedding)}")
                return []

            query_cache[cache_key] = embedding
            return embedding

        except Exception as e:
            logger.error(f"❌ Error generating embedding for {model_name}: {e}. Retrying...")

    logger.error(f"❌ Embedding generation failed for {model_name} after {max_retries + 1} attempts.")
    return []

# ✅ Process Single File (Sequential)
def process_single_file(obj, model_name):
    """
    Reads a single JSON file, generates document-level embedding, and saves back.
    """
    bucket = CONFIG["bucket_name"]
    file_key = obj["Key"]

    logger.info(f"📂 Processing file: {file_key}")

    file_content = read_s3_json(bucket, file_key)
    if file_content is None:
        logger.error(f"❌ Failed to read {file_key}. Skipping.")
        return

    # ✅ Combine text from all chunks
    combined_text = " ".join(clean_text(chunk.get("text", "").strip()) for chunk in file_content.get("chunks", []))

    if not combined_text:
        logger.warning(f"⚠️ No text found in {file_key}. Skipping.")
        return

    # ✅ Generate embedding
    embedding_result = generate_embedding(combined_text, model_name)
    if not embedding_result:
        logger.error(f"❌ No embedding generated for {file_key}. Skipping write.")
        return

    # ✅ Store embedding in JSON
    file_content.setdefault("embeddings", {})[model_name] = embedding_result

    # ✅ Save updated JSON
    save_s3_json(bucket, file_key, file_content)

# ✅ Process All Files (Sequential)
def process_json_files_sequential(model_name):
    """
    Processes all JSON files sequentially (no threading).
    """
    start_time = time.time()
    bucket = CONFIG["bucket_name"]
    folder = CONFIG["json_folder"]

    logger.info(f"🚀 Starting SEQUENTIAL processing for model {model_name}...")

    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=folder)
    file_list = response.get("Contents", [])

    if not file_list:
        logger.error("❌ No JSON files found in S3 bucket. Ensure the folder is populated.")
        return

    total_files = len(file_list)
    logger.info(f"🔍 Found {total_files} files. Processing one at a time...")

    for obj in file_list:
        process_single_file(obj, model_name)

    duration = time.time() - start_time
    logger.info(f"✅ Finished processing {total_files} JSON files in {duration:.2f} seconds.")

# ✅ Run Processing
if __name__ == "__main__":
    selected_model = CONFIG["default_model"]  # Change if needed
    process_json_files_sequential(selected_model)


INFO:__main__:🚀 Starting SEQUENTIAL processing for model amazon.titan-embed-text-v2:0...
INFO:__main__:🔍 Found 297 files. Processing one at a time...
INFO:__main__:📂 Processing file: dev/json/
ERROR:__main__:❌ Failed to read dev/json/. Skipping.
INFO:__main__:📂 Processing file: dev/json/1806.00663_page_1.json
INFO:__main__:⚡ Requesting embedding from AWS Bedrock for amazon.titan-embed-text-v2:0 (Attempt 1)...
INFO:__main__:🕒 Bedrock embedding request took 0.10 seconds.
ERROR:__main__:❌ AWS Bedrock response for amazon.titan-embed-text-v2:0 is missing 'Body'. Retrying...
INFO:__main__:⚡ Requesting embedding from AWS Bedrock for amazon.titan-embed-text-v2:0 (Attempt 2)...
INFO:__main__:🕒 Bedrock embedding request took 0.10 seconds.
ERROR:__main__:❌ AWS Bedrock response for amazon.titan-embed-text-v2:0 is missing 'Body'. Retrying...
ERROR:__main__:❌ Embedding generation failed for amazon.titan-embed-text-v2:0 after 2 attempts.
ERROR:__main__:❌ No embedding generated for dev/json/1806.00663